<a href="https://colab.research.google.com/github/nellypoghosyan/ML-FlyRank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

**Lane:** Refresh / Content Opportunity Scoring  
**Development month:** March 2026  
**Decision moment:** end of 2026-03-21  
**Feature window:** 2026-03-01 → 2026-03-21  
**Outcome window used only for evaluation:** 2026-03-22 → 2026-03-31

This notebook freezes one transparent rule baseline that the Week-5 model must beat.

### Plain-word rule
A page is worth a **refresh review** when:
1. it had meaningful search visibility before the decision moment, and
2. its average search position got worse during the feature window.

The score ranks flagged pages by a simple combination of **how visible the page was** and **how much its position slipped**.

**No future-window or label-derived value is used as a rule input.**

## Setup

The notebook reads the same gated FlyRank warehouse used in Week 3.  
`HF_TOKEN` must exist in **Colab Secrets**. Never paste it into a cell.

In [1]:
!pip -q install duckdb

import os
import json
import duckdb
import pandas as pd
import numpy as np
from IPython.display import display

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Add a Colab Secret named HF_TOKEN and allow notebook access."
    ) from e

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN secret is missing.")

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

BASE = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"{BASE}/fact_content_daily_performance/month=2026-03/*.parquet"

FEATURE_START = "2026-03-01"
EARLY_END = "2026-03-10"
RECENT_START = "2026-03-11"
DECISION_DATE = "2026-03-21"
OUTCOME_START = "2026-03-22"
OUTCOME_END = "2026-03-31"

print("Connected.")
print("Feature window:", FEATURE_START, "to", DECISION_DATE)
print("Outcome window (evaluation only):", OUTCOME_START, "to", OUTCOME_END)

Connected.
Feature window: 2026-03-01 to 2026-03-21
Outcome window (evaluation only): 2026-03-22 to 2026-03-31


## Build the March decision frame

The rule uses only pre-decision fields:

- `impressions_21d` — total GSC impressions from March 1–21.
- `position_early` — impression-weighted average GSC position from March 1–10.
- `position_recent` — impression-weighted average GSC position from March 11–21.
- `position_slip` — `position_recent - position_early`; positive means ranking got worse.

The future decline label is constructed only so I can audit the signals and evaluate the frozen rule. It is **not** a feature.

In [2]:
frame_sql = f"""
WITH agg AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
        ) AS impressions_21d,

        SUM(gsc_clicks) FILTER (
            WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
        ) AS clicks_21d,

        SUM(gsc_avg_position * gsc_impressions) FILTER (
            WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{EARLY_END}'
              AND gsc_impressions > 0
              AND gsc_avg_position > 0
        )
        /
        NULLIF(
            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{EARLY_END}'
                  AND gsc_impressions > 0
                  AND gsc_avg_position > 0
            ),
            0
        ) AS position_early,

        SUM(gsc_avg_position * gsc_impressions) FILTER (
            WHERE report_date BETWEEN DATE '{RECENT_START}' AND DATE '{DECISION_DATE}'
              AND gsc_impressions > 0
              AND gsc_avg_position > 0
        )
        /
        NULLIF(
            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '{RECENT_START}' AND DATE '{DECISION_DATE}'
                  AND gsc_impressions > 0
                  AND gsc_avg_position > 0
            ),
            0
        ) AS position_recent,

        AVG(gsc_impressions) FILTER (
            WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
        ) AS past_avg_daily_impressions,

        AVG(gsc_impressions) FILTER (
            WHERE report_date BETWEEN DATE '{OUTCOME_START}' AND DATE '{OUTCOME_END}'
        ) AS future_avg_daily_impressions

    FROM read_parquet('{MARCH}')
    GROUP BY 1, 2
)

SELECT
    client_hash_id,
    content_hash_id,
    impressions_21d,
    clicks_21d,
    position_early,
    position_recent,
    position_recent - position_early AS position_slip,

    CASE
        WHEN impressions_21d >= 100
         AND past_avg_daily_impressions > 0
         AND future_avg_daily_impressions IS NOT NULL
         AND future_avg_daily_impressions < 0.80 * past_avg_daily_impressions
        THEN 1
        ELSE 0
    END AS future_decline_label

FROM agg
WHERE impressions_21d >= 100
  AND position_early IS NOT NULL
  AND position_recent IS NOT NULL
  AND past_avg_daily_impressions > 0
  AND future_avg_daily_impressions IS NOT NULL
"""

df = con.sql(frame_sql).df()

print("Decision-grain rows:", f"{len(df):,}")
print("Future-decline base rate:", round(df["future_decline_label"].mean(), 4))
display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Decision-grain rows: 84,008
Future-decline base rate: 0.3211


,client_hash_id,content_hash_id,impressions_21d,clicks_21d,position_early,position_recent,position_slip,future_decline_label
0,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,309.0,2.0,3.944882,4.653846,0.708964,0
1,client_62f4a7e64f5e0096,content_275b6f7f733016d4,611.0,1.0,4.357349,4.397727,0.040379,1
2,client_62f4a7e64f5e0096,content_755d951187fcd70a,1188.0,4.0,1.956190,2.069382,0.113191,0
3,client_62f4a7e64f5e0096,content_92c381fbd361212e,359.0,1.0,5.052288,4.024272,-1.028016,0
4,client_62f4a7e64f5e0096,content_97188a7032a705cf,321.0,3.0,3.200000,3.254144,0.054144,0


# 1. Check two signals first

I test the signals **before** encoding the rule.

### Signal A — pre-decision volume
**Claim:** pages with more pre-decision search visibility are more important opportunities and may also show a different future-decline rate.

This is linked to FlyRank's **quick-win / volume** logic. Because traffic is heavy-tailed, I use quantile buckets and always show `n`.

### Signal B — position slipping
**Claim:** pages whose search position worsens during the feature window are more likely to decline in the following outcome window.

For position, **higher = worse**. `position_slip > 0` means the page moved down in search results.

In [3]:
# SIGNAL A — VOLUME BUCKET TABLE

signal_a = df.copy()
signal_a["volume_bucket"] = pd.qcut(
    signal_a["impressions_21d"],
    q=4,
    labels=["Q1_low", "Q2", "Q3", "Q4_high"],
    duplicates="drop"
)

volume_table = (
    signal_a.groupby("volume_bucket", observed=True)
    .agg(
        n=("future_decline_label", "size"),
        median_impressions=("impressions_21d", "median"),
        decline_rate=("future_decline_label", "mean")
    )
    .reset_index()
)

display(volume_table)

low = volume_table.iloc[0]
high = volume_table.iloc[-1]
diff = high["decline_rate"] - low["decline_rate"]

if min(low["n"], high["n"]) < 50:
    volume_verdict = "MIXED"
    volume_note = "Edge buckets are too small for a confident direction."
elif diff >= 0.03:
    volume_verdict = "CONFIRMED"
    volume_note = "The highest-volume bucket has a meaningfully higher decline rate than the lowest."
elif diff <= -0.03:
    volume_verdict = "OPPOSITE"
    volume_note = "The highest-volume bucket has a meaningfully lower decline rate than the lowest."
elif abs(diff) < 0.015:
    volume_verdict = "FALSE"
    volume_note = "Volume changes priority/impact, but it shows almost no directional decline-risk signal here."
else:
    volume_verdict = "MIXED"
    volume_note = "The difference is small or not clean enough to call directional."

print("VOLUME VERDICT:", volume_verdict)
print(volume_note)

,volume_bucket,n,median_impressions,decline_rate
0,Q1_low,21002,163.0,0.373488
1,Q2,21028,427.0,0.355669
2,Q3,20981,1154.0,0.294981
3,Q4_high,20997,4009.0,0.260180


VOLUME VERDICT: OPPOSITE
The highest-volume bucket has a meaningfully lower decline rate than the lowest.


In [4]:
# SIGNAL B — POSITION-SLIP BUCKET TABLE

def slip_bucket(x):
    if x <= -2:
        return "improved_2plus"
    elif x < 2:
        return "roughly_stable"
    elif x < 5:
        return "slipped_2_to_5"
    else:
        return "slipped_5plus"

signal_b = df.copy()
signal_b["slip_bucket"] = signal_b["position_slip"].map(slip_bucket)

order = ["improved_2plus", "roughly_stable", "slipped_2_to_5", "slipped_5plus"]
signal_b["slip_bucket"] = pd.Categorical(signal_b["slip_bucket"], categories=order, ordered=True)

slip_table = (
    signal_b.groupby("slip_bucket", observed=True)
    .agg(
        n=("future_decline_label", "size"),
        median_position_slip=("position_slip", "median"),
        decline_rate=("future_decline_label", "mean")
    )
    .reset_index()
)

display(slip_table)

stable_rows = slip_table[slip_table["slip_bucket"] == "roughly_stable"]
slipped_rows = slip_table[slip_table["slip_bucket"] == "slipped_5plus"]

if stable_rows.empty or slipped_rows.empty:
    slip_verdict = "MIXED"
    slip_note = "A required comparison bucket is empty."
else:
    stable = stable_rows.iloc[0]
    slipped = slipped_rows.iloc[0]
    diff = slipped["decline_rate"] - stable["decline_rate"]

    if min(stable["n"], slipped["n"]) < 50:
        slip_verdict = "MIXED"
        slip_note = "One comparison bucket is below the sample-size floor."
    elif diff >= 0.05:
        slip_verdict = "CONFIRMED"
        slip_note = "Large position slips are followed by a clearly higher decline rate than stable positions."
    elif diff <= -0.05:
        slip_verdict = "OPPOSITE"
        slip_note = "Large position slips are followed by a lower decline rate than stable positions."
    elif abs(diff) < 0.02:
        slip_verdict = "FALSE"
        slip_note = "Large position slips do not separate future decline risk in this slice."
    else:
        slip_verdict = "MIXED"
        slip_note = "The direction exists but is not strong/clean enough for a confirmed verdict."

print("POSITION-SLIP VERDICT:", slip_verdict)
print(slip_note)

,slip_bucket,n,median_position_slip,decline_rate
0,improved_2plus,18797,-4.517093,0.296377
1,roughly_stable,44840,0.041460,0.295540
2,slipped_2_to_5,9795,3.115586,0.367024
3,slipped_5plus,10576,9.213937,0.430881


POSITION-SLIP VERDICT: CONFIRMED
Large position slips are followed by a clearly higher decline rate than stable positions.


## Rule reasoning after the signal checks

I keep the baseline intentionally simple:

> **Flag for refresh review if the page had at least 500 pre-decision impressions and its position worsened by at least 2 positions. Rank flagged pages by visibility × amount of slippage.**

Volume is mainly an **impact/prioritization** signal. Even if its verdict is MIXED/FALSE as a predictor of decline, it can still reasonably decide which risky page deserves attention first.

There are **no fitted weights**.

# 2. Encode ONE rule and build the ranked queue

Required fields:

- `baseline_score`
- exactly one reason code: `visible_position_slipping`
- one action label: `REVIEW_REFRESH`

The CSV is regenerated on every run at:

`work/outputs/baseline_action_score.csv`

In [5]:
# Freeze the one transparent rule.

MIN_IMPRESSIONS = 500
MIN_POSITION_SLIP = 2.0

rule_input_columns = [
    "impressions_21d",
    "position_early",
    "position_recent",
    "position_slip",
]

# Explicit leakage guard: only these pre-decision inputs can drive the rule.
FORBIDDEN_INPUTS = {
    "future_decline_label",
    "future_avg_daily_impressions",
    "future_drop_ratio",
    "trend_pct",
    "trend_direction",
}

assert not (set(rule_input_columns) & FORBIDDEN_INPUTS)

scored = df.copy()

scored["is_flagged"] = (
    (scored["impressions_21d"] >= MIN_IMPRESSIONS)
    & (scored["position_slip"] >= MIN_POSITION_SLIP)
)

# Simple, transparent score: impact × risk magnitude.
scored["baseline_score"] = np.where(
    scored["is_flagged"],
    np.log1p(scored["impressions_21d"]) * scored["position_slip"],
    0.0
)

queue = (
    scored.loc[scored["is_flagged"]]
    .assign(
        reason_code="visible_position_slipping",
        action_label="REVIEW_REFRESH"
    )
    .sort_values(["baseline_score", "impressions_21d"], ascending=False)
    .reset_index(drop=True)
)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

print("Flagged rows:", f"{len(queue):,}")
print("One reason code:", queue["reason_code"].unique().tolist())
print("One action label:", queue["action_label"].unique().tolist())

display(
    queue[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "baseline_score",
            "impressions_21d",
            "position_early",
            "position_recent",
            "position_slip",
            "reason_code",
            "action_label",
        ]
    ].head(10)
)

Flagged rows: 8,032
One reason code: ['visible_position_slipping']
One action label: ['REVIEW_REFRESH']


,rank,client_hash_id,content_hash_id,baseline_score,impressions_21d,position_early,position_recent,position_slip,reason_code,action_label
0,1,client_3197e6291363b4db,content_2392ac0360e7c84c,391.662608,644.0,14.666667,75.208861,60.542194,visible_position_slipping,REVIEW_REFRESH
1,2,client_73cda7b4e4f265ea,content_e98ee386179e4124,359.715710,706.0,0.272494,55.098592,54.826098,visible_position_slipping,REVIEW_REFRESH
2,3,client_73cda7b4e4f265ea,content_5a7268c256a7fd98,346.395924,3196.0,10.925783,53.849858,42.924076,visible_position_slipping,REVIEW_REFRESH
3,4,client_73cda7b4e4f265ea,content_78ed91c540dcffd0,345.334646,522.0,8.347162,63.516129,55.168967,visible_position_slipping,REVIEW_REFRESH
4,5,client_23a62021009f63c4,content_7970c02ba39fd988,343.587307,630.0,1.969697,55.261307,53.291610,visible_position_slipping,REVIEW_REFRESH
5,6,client_23a62021009f63c4,content_af30df68884a76e2,318.601275,764.0,7.142857,55.125874,47.983017,visible_position_slipping,REVIEW_REFRESH
6,7,client_73cda7b4e4f265ea,content_9bcfb1e373c01b7a,299.953248,9174.0,1.949960,34.824295,32.874335,visible_position_slipping,REVIEW_REFRESH
7,8,client_08a6a72ff48e62c0,content_a62ac535920e8a80,298.335359,654.0,33.304786,79.311284,46.006498,visible_position_slipping,REVIEW_REFRESH
8,9,client_73cda7b4e4f265ea,content_1288165df2e387d6,297.407742,3300.0,9.401283,46.109312,36.708029,visible_position_slipping,REVIEW_REFRESH
9,10,client_73cda7b4e4f265ea,content_1ab1a39d23558ea3,297.133826,3874.0,12.829292,48.791892,35.962600,visible_position_slipping,REVIEW_REFRESH


In [6]:
# Baseline evaluation receipt.
# The future label is used ONLY here to evaluate the frozen queue, never to construct its score.

def precision_at_k(ranked_df, label_col, k):
    if len(ranked_df) == 0:
        return np.nan
    k = min(k, len(ranked_df))
    return ranked_df.head(k)[label_col].mean()

base_rate = df["future_decline_label"].mean()
p10 = precision_at_k(queue, "future_decline_label", 10)
p50 = precision_at_k(queue, "future_decline_label", 50)

metrics = {
    "lane": "Refresh / Content Opportunity Scoring",
    "development_month": "2026-03",
    "feature_window": f"{FEATURE_START} to {DECISION_DATE}",
    "outcome_window_evaluation_only": f"{OUTCOME_START} to {OUTCOME_END}",
    "base_rate": float(base_rate),
    "flagged_rows": int(len(queue)),
    "precision_at_10": None if np.isnan(p10) else float(p10),
    "precision_at_50": None if np.isnan(p50) else float(p50),
    "volume_signal_verdict": volume_verdict,
    "position_slip_signal_verdict": slip_verdict,
}

print("Base rate:", round(base_rate, 4))
print("Precision@10:", None if np.isnan(p10) else round(p10, 4))
print("Precision@50:", None if np.isnan(p50) else round(p50, 4))

Base rate: 0.3211
Precision@10: 0.8
Precision@50: 0.5


In [7]:
# Write the required queue CSV.
# It stays out of git by design; the notebook regenerates it.

os.makedirs("work/outputs", exist_ok=True)

csv_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "impressions_21d",
    "position_early",
    "position_recent",
    "position_slip",
    "reason_code",
    "action_label",
]

csv_path = "work/outputs/baseline_action_score.csv"
queue[csv_cols].to_csv(csv_path, index=False)

json_path = "work/outputs/w04_baseline_metrics.json"
with open(json_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Wrote:", csv_path)
print("Wrote metrics receipt:", json_path)

Wrote: work/outputs/baseline_action_score.csv
Wrote metrics receipt: work/outputs/w04_baseline_metrics.json


# 3. Top-10 skeptical review

For each of the top ten I record:

- **action**
- **why it is there**
- **what would make it wrong**

The last part matters: a high score is a review priority, not proof that refreshing the page will help.

In [8]:
top10 = queue.head(10).copy()

def skeptic_line(row):
    wrong_if = (
        "the position move is temporary/noisy, the query mix changed, "
        "or the page is already being intentionally retired"
    )
    return (
        f"#{int(row['rank'])} {row['action_label']} — "
        f"why: {int(row['impressions_21d']):,} pre-decision impressions and "
        f"position worsened by {row['position_slip']:.1f} places "
        f"({row['position_early']:.1f} → {row['position_recent']:.1f}); "
        f"wrong if: {wrong_if}."
    )

review_lines = [skeptic_line(row) for _, row in top10.iterrows()]

for line in review_lines:
    print(line)

#1 REVIEW_REFRESH — why: 644 pre-decision impressions and position worsened by 60.5 places (14.7 → 75.2); wrong if: the position move is temporary/noisy, the query mix changed, or the page is already being intentionally retired.
#2 REVIEW_REFRESH — why: 706 pre-decision impressions and position worsened by 54.8 places (0.3 → 55.1); wrong if: the position move is temporary/noisy, the query mix changed, or the page is already being intentionally retired.
#3 REVIEW_REFRESH — why: 3,196 pre-decision impressions and position worsened by 42.9 places (10.9 → 53.8); wrong if: the position move is temporary/noisy, the query mix changed, or the page is already being intentionally retired.
#4 REVIEW_REFRESH — why: 522 pre-decision impressions and position worsened by 55.2 places (8.3 → 63.5); wrong if: the position move is temporary/noisy, the query mix changed, or the page is already being intentionally retired.
#5 REVIEW_REFRESH — why: 630 pre-decision impressions and position worsened by 53.3 

In [9]:
# Structured top-10 review table too, so the evidence is easy to scan.

top10_review = top10[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action_label",
        "reason_code",
        "impressions_21d",
        "position_early",
        "position_recent",
        "position_slip",
        "future_decline_label",
    ]
].copy()

top10_review["why_it_is_here"] = top10_review.apply(
    lambda r: (
        f"{int(r['impressions_21d']):,} impressions; "
        f"position slipped {r['position_slip']:.1f}"
    ),
    axis=1
)

top10_review["what_would_make_it_wrong"] = (
    "Temporary/noisy position change, changed query mix, or intentionally retired content."
)

display(top10_review)

,rank,client_hash_id,content_hash_id,action_label,reason_code,impressions_21d,position_early,position_recent,position_slip,future_decline_label,why_it_is_here,what_would_make_it_wrong
0,1,client_3197e6291363b4db,content_2392ac0360e7c84c,REVIEW_REFRESH,visible_position_slipping,644.0,14.666667,75.208861,60.542194,0,644 impressions; position slipped 60.5,"Temporary/noisy position change, changed query..."
1,2,client_73cda7b4e4f265ea,content_e98ee386179e4124,REVIEW_REFRESH,visible_position_slipping,706.0,0.272494,55.098592,54.826098,1,706 impressions; position slipped 54.8,"Temporary/noisy position change, changed query..."
2,3,client_73cda7b4e4f265ea,content_5a7268c256a7fd98,REVIEW_REFRESH,visible_position_slipping,3196.0,10.925783,53.849858,42.924076,1,"3,196 impressions; position slipped 42.9","Temporary/noisy position change, changed query..."
3,4,client_73cda7b4e4f265ea,content_78ed91c540dcffd0,REVIEW_REFRESH,visible_position_slipping,522.0,8.347162,63.516129,55.168967,1,522 impressions; position slipped 55.2,"Temporary/noisy position change, changed query..."
4,5,client_23a62021009f63c4,content_7970c02ba39fd988,REVIEW_REFRESH,visible_position_slipping,630.0,1.969697,55.261307,53.291610,1,630 impressions; position slipped 53.3,"Temporary/noisy position change, changed query..."
5,6,client_23a62021009f63c4,content_af30df68884a76e2,REVIEW_REFRESH,visible_position_slipping,764.0,7.142857,55.125874,47.983017,0,764 impressions; position slipped 48.0,"Temporary/noisy position change, changed query..."
6,7,client_73cda7b4e4f265ea,content_9bcfb1e373c01b7a,REVIEW_REFRESH,visible_position_slipping,9174.0,1.949960,34.824295,32.874335,1,"9,174 impressions; position slipped 32.9","Temporary/noisy position change, changed query..."
7,8,client_08a6a72ff48e62c0,content_a62ac535920e8a80,REVIEW_REFRESH,visible_position_slipping,654.0,33.304786,79.311284,46.006498,1,654 impressions; position slipped 46.0,"Temporary/noisy position change, changed query..."
8,9,client_73cda7b4e4f265ea,content_1288165df2e387d6,REVIEW_REFRESH,visible_position_slipping,3300.0,9.401283,46.109312,36.708029,1,"3,300 impressions; position slipped 36.7","Temporary/noisy position change, changed query..."
9,10,client_73cda7b4e4f265ea,content_1ab1a39d23558ea3,REVIEW_REFRESH,visible_position_slipping,3874.0,12.829292,48.791892,35.962600,1,"3,874 impressions; position slipped 36.0","Temporary/noisy position change, changed query..."


# 4. Weak picks + leakage check

A baseline should expose its weaknesses.

I look for top-10 rows where the future decline label is `0`. Those are not automatically “bad” recommendations — the future label is only a proxy — but they are useful cases to question.

### Known weaknesses of this rule
- Position can move because the **query mix changes**, not because the page itself got worse.
- High-volume pages dominate priority even when the volume signal is not a strong predictor of decline.
- The rule does not know whether content was already scheduled for retirement, consolidation, or another product decision.
- It does not prove that a refresh would cause recovery.

In [10]:
weak_picks = top10_review[top10_review["future_decline_label"] == 0]

print("Top-10 picks not positive under the future-decline proxy:", len(weak_picks))
if len(weak_picks):
    display(
        weak_picks[
            [
                "rank",
                "content_hash_id",
                "impressions_21d",
                "position_slip",
                "future_decline_label",
                "what_would_make_it_wrong",
            ]
        ]
    )
else:
    print(
        "None in this top 10. That does NOT prove the rule is perfect; "
        "the proxy itself is limited, so I still keep the skeptical notes above."
    )

print("\nLEAKAGE CHECK")
print("Rule inputs:", rule_input_columns)
print("Forbidden inputs used by rule:", sorted(set(rule_input_columns) & FORBIDDEN_INPUTS))
print("PASS:", len(set(rule_input_columns) & FORBIDDEN_INPUTS) == 0)

Top-10 picks not positive under the future-decline proxy: 2


,rank,content_hash_id,impressions_21d,position_slip,future_decline_label,what_would_make_it_wrong
0,1,content_2392ac0360e7c84c,644.0,60.542194,0,"Temporary/noisy position change, changed query..."
5,6,content_af30df68884a76e2,764.0,47.983017,0,"Temporary/noisy position change, changed query..."



LEAKAGE CHECK
Rule inputs: ['impressions_21d', 'position_early', 'position_recent', 'position_slip']
Forbidden inputs used by rule: []
PASS: True


## Lane confirmation

**CONFIRMED lane:** Refresh / Content Opportunity Scoring.

I am keeping this lane for Week 5. The frozen baseline is the queue produced by the rule above; I will not move its thresholds after seeing the model result just to make the comparison easier.

# 5. Self-check

After **Runtime → Run all**, confirm:

- [ ] Two signal checks are visible.
- [ ] Each signal has a bucket table with `n`.
- [ ] At least one signal is tied to a real FlyRank flag family (volume / quick-win).
- [ ] Each signal prints one verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE.
- [ ] The rule is stated in plain words.
- [ ] Exactly one reason code is used: `visible_position_slipping`.
- [ ] Exactly one action label is used: `REVIEW_REFRESH`.
- [ ] A ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [ ] Base rate and precision@K are printed.
- [ ] Ten top-ranked rows each have action, why, and what would make it wrong.
- [ ] Weak picks are inspected.
- [ ] The leakage check prints `PASS: True`.
- [ ] No future-window or label-derived value is an input to the baseline score.
- [ ] The lane is explicitly confirmed.
- [ ] Notebook is saved as `work/notebooks/w04_baseline_score.ipynb`.